# SDOBenchmark — data exploration

Goal: understand the dataset before training anything. Run the cells top to bottom.

**Vocabulary used everywhere below**

- **Active region (AR)**: one patch of the Sun with a tangled magnetic field. It has a NOAA number (e.g. `12673`) and lives for days.
- **Sample**: one *moment* in the life of one active region: 12 h of images (4 time steps x 10 channels) plus one label, the strongest flare in the following 24 h. One region gives many samples.
- **Flaring region** (for our task): an active region that has at least one sample labelled M or X.

On Kaggle: *Add Input* -> search `sdobenchmark` (fhnw-i4ds), then run. The first cell finds the data by itself.

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")

# ---- find the dataset (works on Kaggle, Colab and locally) ------------------
DATA_ROOT = None          # set a path here if auto-detection fails

def _find(bases):
    for base in bases:
        hits = glob.glob(os.path.join(base, "**", "training", "meta_data.csv"), recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(sorted(hits, key=len)[0]))

if DATA_ROOT is None:
    DATA_ROOT = _find(["/kaggle/input", ".", ".."])
if DATA_ROOT is None:     # e.g. Google Colab: download from Kaggle (~7 GB, onto the Colab machine, not your PC)
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    path = kagglehub.dataset_download("fhnw-i4ds/sdobenchmark")
    print("downloaded to", path)
    DATA_ROOT = _find([path])
assert DATA_ROOT, "Dataset not found: set DATA_ROOT to the folder that contains training/ and test/"
print("DATA_ROOT =", DATA_ROOT)

M_THRESHOLD = 1e-5        # peak flux of an M1.0 flare (W/m^2)
C_THRESHOLD = 1e-6
HMI = ["magnetogram", "continuum"]
AIA = ["1700", "304", "171", "193", "211", "335", "94", "131"]
CHANNELS = HMI + AIA
STEP_NAMES = ["12 h before", "5 h before", "1.5 h before", "10 min before"]

## 1. The labels file

Each split (`training`, `test`) has a `meta_data.csv` with one row per sample:

| column | meaning |
|---|---|
| `id` | `{active region}_{date}_{n}`; the part before the first `_` is the region number |
| `start`, `end` | the 12 h during which the input images were taken |
| `peak_flux` | strongest X-ray flux in the **24 h after `end`**. `1e-9` means nothing happened |

We add three columns: `ar` (region), `cls` (flare letter) and `y` (our binary label, flare >= M).

In [ ]:
def flux_to_class(f):
    if f < 1e-7: return "quiet"
    return "B" if f < 1e-6 else "C" if f < 1e-5 else "M" if f < 1e-4 else "X"

def goes_name(f):
    if f < 1e-7: return "quiet"
    for letter, base in [("X", 1e-4), ("M", 1e-5), ("C", 1e-6), ("B", 1e-7)]:
        if f >= base: return f"{letter}{f / base:.1f}"

def load_meta(split):
    m = pd.read_csv(os.path.join(DATA_ROOT, split, "meta_data.csv"), parse_dates=["start", "end"])
    m["split"] = split
    m["ar"] = m["id"].str.split("_").str[0]
    m["cls"] = m["peak_flux"].map(flux_to_class)
    m["y"] = (m["peak_flux"] >= M_THRESHOLD).astype(int)
    m["dir"] = [os.path.join(DATA_ROOT, split, a, i.split("_", 1)[1]) for a, i in zip(m["ar"], m["id"])]
    return m

meta = pd.concat([load_meta("training"), load_meta("test")], ignore_index=True)
meta.head()

## 2. Samples vs active regions vs flaring regions

Three different counts. Mixing them up is the most common mistake with this dataset.

In [ ]:
rows = []
for split, m in meta.groupby("split", sort=False):
    per_ar = m.groupby("ar").agg(n=("id", "size"), n_pos=("y", "sum"), max_flux=("peak_flux", "max"))
    flaring = per_ar[per_ar.n_pos > 0]
    rows.append({
        "split": split,
        "samples": len(m),
        "active regions": len(per_ar),
        "samples >= M (positives)": int(m.y.sum()),
        "flaring regions (>= 1 positive)": len(flaring),
        "samples inside flaring regions": int(flaring.n.sum()),
        "... of which NOT positive": int(flaring.n.sum() - flaring.n_pos.sum()),
        "regions that never leave 'quiet'": int((per_ar.max_flux < 1e-7).sum()),
    })
pd.DataFrame(rows).set_index("split").T

Read the table like this:

- **samples** is what the model sees as rows. **active regions** is how many *independent* things you really have.
- All positives come from the small group of **flaring regions**.
- A flaring region is not always flaring: look at "of which NOT positive". The same region gives quiet, C and M samples at different moments. So the model cannot just learn "this region is dangerous"; it has to tell *when*.

Next cell: the life of one region, sample by sample.

In [ ]:
def plot_region(ar):
    a = meta[meta.ar == ar].sort_values("end")
    fig, ax = plt.subplots(figsize=(11, 3.2))
    colors = a.cls.map({"quiet": "#888780", "B": "#378ADD", "C": "#1D9E75", "M": "#EF9F27", "X": "#E24B4A"})
    ax.scatter(a.end, a.peak_flux, c=colors, s=60, zorder=3)
    ax.set_yscale("log"); ax.set_ylim(5e-10, 3e-3)
    ax.axhline(M_THRESHOLD, ls="--", lw=1, color="k"); ax.text(a.end.min(), M_THRESHOLD * 1.3, "M threshold: above = positive", fontsize=9)
    ax.set_ylabel("peak flux in next 24 h"); ax.set_xlabel("end of the input window")
    ax.set_title(f"Active region {ar} ({a.split.iloc[0]}): {len(a)} samples, {int(a.y.sum())} positive.  One dot = one sample")
    plt.tight_layout(); plt.show()
    return a[["id", "end", "cls", "peak_flux", "y"]]

# the region with the most positive samples; replace by any number, e.g. "12673"
busiest = meta.groupby("ar").y.sum().idxmax()
plot_region(busiest)

In [ ]:
# how unevenly are samples and positives spread over regions?
per_ar = meta.groupby(["split", "ar"]).agg(n=("id", "size"), n_pos=("y", "sum")).reset_index()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for split, g in per_ar.groupby("split"):
    ax[0].hist(g.n, bins=range(1, g.n.max() + 2), alpha=.6, label=split)
    pos = g.n_pos.sort_values(ascending=False).values
    if pos.sum(): ax[1].plot(np.arange(1, len(pos) + 1), np.cumsum(pos) / pos.sum(), label=split)
ax[0].set_xlabel("samples per region"); ax[0].set_ylabel("number of regions"); ax[0].set_yscale("log"); ax[0].legend()
ax[1].set_xlabel("regions, sorted by number of positives"); ax[1].set_ylabel("share of all positive samples"); ax[1].set_xscale("log"); ax[1].legend()
ax[1].set_title("A few regions own most positives")
plt.tight_layout(); plt.show()

## 3. Class balance

`quiet < B < C < M < X`, each letter 10x stronger. Our positive class is M + X.

In [ ]:
order = ["quiet", "B", "C", "M", "X"]
counts = meta.groupby(["split", "cls"]).size().unstack(fill_value=0).reindex(columns=order, fill_value=0)
counts["positives (M+X)"] = counts["M"] + counts["X"]
counts["% positive"] = (100 * counts["positives (M+X)"] / counts[order].sum(axis=1)).round(1)
display(counts)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for split, g in meta.groupby("split"):
    ax[0].hist(np.log10(g.peak_flux), bins=40, alpha=.6, label=split, density=True)
ax[0].axvline(np.log10(M_THRESHOLD), ls="--", color="k"); ax[0].set_xlabel("log10(peak_flux)"); ax[0].set_ylabel("density"); ax[0].legend()
by_year = meta.assign(year=meta.end.dt.year).groupby(["year", "split"]).y.sum().unstack(fill_value=0)
by_year.plot.bar(ax=ax[1]); ax[1].set_ylabel("positive samples"); ax[1].set_title("Positives per year (solar cycle)")
plt.tight_layout(); plt.show()

Two things to remember:

- The **positive rate differs** between training and test. TSS does not care, but Brier score, F1 and any decision threshold do.
- Flares follow the **solar cycle**: late years have very few. A split by date would give a validation set with almost no positives. Split by region instead (section 8).

## 4. Open one sample

A sample folder holds up to 40 JPEGs named `{timestamp}__{channel}.jpg`.

In [ ]:
def sample_files(d):
    """dict {(step_index, channel): path}; step 0 = oldest, 3 = 10 min before."""
    files = glob.glob(os.path.join(d, "*.jpg"))
    stamps = sorted({os.path.basename(p).split("__")[0] for p in files})
    return {(stamps.index(os.path.basename(p).split("__")[0]), os.path.basename(p)[:-4].split("__")[1]): p for p in files}, stamps

def read(path):
    return np.asarray(Image.open(path)) if path and os.path.exists(path) else None

def show_sample(row):
    files, stamps = sample_files(row.dir)
    fig, ax = plt.subplots(len(stamps), 10, figsize=(20, 2.15 * len(stamps)), squeeze=False)
    for i, s in enumerate(stamps):
        for j, c in enumerate(CHANNELS):
            a = read(files.get((i, c))); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if a is not None: ax[i, j].imshow(a, cmap="gray", vmin=0, vmax=255)
            else: ax[i, j].text(.5, .5, "missing", ha="center", va="center", color="red")
            if i == 0: ax[i, j].set_title(("HMI " if c in HMI else "AIA ") + c)
            if j == 0: ax[i, j].set_ylabel(s[5:])
    fig.suptitle(f"Region {row.ar} | input ends {row.end:%Y-%m-%d %H:%M} | next 24 h: {goes_name(row.peak_flux)}", fontsize=14)
    plt.tight_layout(); plt.show()

show_sample(meta.sort_values("peak_flux").iloc[-1])      # strongest flare in the data

What each channel is, in plain words:

| channel | what you see | why it matters |
|---|---|---|
| HMI `magnetogram` | magnetic field: white = one polarity, black = the other, grey = none | the *cause*. Black and white tightly mixed = stressed field |
| HMI `continuum` | the visible surface, sunspots are dark | size and shape of the spots |
| AIA `1700` | lower atmosphere | bright network around the spots |
| AIA `304` | chromosphere, about 50,000 K | filaments, eruptions starting |
| AIA `171`, `193`, `211` | corona, about 1-2 million K | magnetic loops |
| AIA `335`, `94`, `131` | hot corona, several million K and more | plasma heated by flaring: the *symptom* |

Temperatures are approximate textbook values. Try other samples:

In [ ]:
show_sample(meta[meta.cls == "quiet"].sample(1, random_state=1).iloc[0])
show_sample(meta[meta.cls == "M"].sample(1, random_state=1).iloc[0])

## 5. Quiet versus flaring, side by side

Last time step only (10 min before the forecast window), four channels.

In [ ]:
def last_step(row, channel):
    files, stamps = sample_files(row.dir)
    return read(files.get((len(stamps) - 1, channel)))

def compare(classes=("quiet", "quiet", "C", "C", "M", "M", "X", "X"), channels=("magnetogram", "171", "94", "131"), seed=0):
    rng = np.random.default_rng(seed)
    picks = []
    for c in classes:
        pool = meta[(meta.cls == c) & (~meta.ar.isin([p.ar for p in picks]))]      # a different region each time
        if len(pool): picks.append(pool.iloc[rng.integers(len(pool))])
    fig, ax = plt.subplots(len(channels), len(picks), figsize=(2.5 * len(picks), 2.55 * len(channels)), squeeze=False)
    for j, r in enumerate(picks):
        for i, c in enumerate(channels):
            a = last_step(r, c); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if a is not None: ax[i, j].imshow(a, cmap="gray", vmin=0, vmax=255)
            if i == 0: ax[i, j].set_title(f"{goes_name(r.peak_flux)}\nAR {r.ar}")
            if j == 0: ax[i, j].set_ylabel(c)
    plt.tight_layout(); plt.show()

compare(seed=0)

Change `seed` and look at 20-30 examples. Questions to answer for yourself:

- Can *you* tell M from quiet on the magnetogram alone? On AIA 94 alone?
- Do some inputs already show a flare in progress (saturated white, spikes)? Then the model can score by "it is flaring now".

## 6. Why you must never split randomly

Several samples from the **same region** look almost the same. If one is in training and another in validation, the model is tested on something it has nearly seen.

In [ ]:
ar = meta.groupby("ar").size().idxmax()            # region with the most samples
a = meta[meta.ar == ar].sort_values("end")
pick = a.iloc[np.linspace(0, len(a) - 1, min(8, len(a))).astype(int)]
fig, ax = plt.subplots(1, len(pick), figsize=(2.5 * len(pick), 2.9), squeeze=False)
for j, r in enumerate(pick.itertuples()):
    img = last_step(r, "magnetogram"); ax[0, j].set_xticks([]); ax[0, j].set_yticks([])
    if img is not None: ax[0, j].imshow(img, cmap="gray", vmin=0, vmax=255)
    ax[0, j].set_title(f"{r.end:%m-%d %H:%M}\n{goes_name(r.peak_flux)}", fontsize=10)
fig.suptitle(f"Region {ar}: {len(a)} samples, magnetogram over time"); plt.tight_layout(); plt.show()

## 7. Missing, blank and flagged images

Three ways an image can be unusable:

1. **missing**: the file is not there
2. **blank**: the file exists but is uniformly black
3. **flagged**: the satellite marked it as noisy (eclipse, calibration); the word `flagged` is in the JPEG metadata

Part A counts files (fast, whole dataset). Part B opens images (slower, on a random subset).

In [ ]:
# Part A: which (step, channel) files exist, for every sample
def presence(d):
    files, stamps = sample_files(d)
    return {"n_images": len(files), "n_steps": len(stamps),
            **{f"last_{c}": (len(stamps) - 1, c) in files for c in CHANNELS}}

pres = pd.DataFrame([presence(d) for d in meta.dir])
meta = pd.concat([meta.drop(columns=[c for c in pres.columns if c in meta.columns]), pres], axis=1)
print("samples with all 40 images:", f"{100 * (meta.n_images == 40).mean():.1f}%")
print("samples with fewer than 4 time steps:", int((meta.n_steps < 4).sum()))
print("\n% of samples that have each channel at the LAST time step:")
display((100 * meta.groupby("split")[[f"last_{c}" for c in CHANNELS]].mean()).round(1).rename(columns=lambda c: c[5:]))
usable = meta.last_magnetogram & meta.last_94 & meta.last_131
print(f"usable for 'last step, magnetogram + 94 + 131': {usable.sum()} of {len(meta)} samples, {int(meta[usable].y.sum())} of {int(meta.y.sum())} positives")

In [ ]:
# Part B: open images to detect blank and flagged ones (subset; raise N_CHECK for a full audit)
N_CHECK = 300
rows = []
for r in meta.sample(min(N_CHECK, len(meta)), random_state=0).itertuples():
    files, _ = sample_files(r.dir)
    for (step, ch), p in files.items():
        im = Image.open(p); a = np.asarray(im)
        info = (str(im.info) + str(dict(im.getexif()))).lower()
        rows.append((ch, a.std() < 3, "flag" in info))
q = pd.DataFrame(rows, columns=["channel", "blank", "flagged"])
print(f"{len(q)} images checked | blank: {100 * q.blank.mean():.2f}% | flagged: {100 * q.flagged.mean():.2f}%")
display((100 * q.groupby("channel")[["blank", "flagged"]].mean()).round(2).reindex(CHANNELS))

**Consequence for your loader**: before using an image, check that the file exists and that `std > 3`. Decide one rule and keep it for all models (simplest: drop the sample if a channel you need is missing at the last step).

## 8. Pixel values and how to normalise

The images are 8-bit (0-255), already clipped and scaled by the dataset authors.

In [ ]:
vals = {c: [] for c in CHANNELS}
for r in meta.sample(min(150, len(meta)), random_state=1).itertuples():
    for c in CHANNELS:
        a = last_step(r, c)
        if a is not None and a.std() > 3: vals[c].append(a.ravel()[::7])
stats = pd.DataFrame({c: {"mean": np.concatenate(v).mean(), "std": np.concatenate(v).std(),
                          "% at 0": 100 * (np.concatenate(v) == 0).mean(), "% at 255": 100 * (np.concatenate(v) == 255).mean()}
                      for c, v in vals.items() if v}).T.round(1)
display(stats)

fig, ax = plt.subplots(2, 5, figsize=(18, 5))
for k, c in enumerate(CHANNELS):
    if vals[c]: ax.flat[k].hist(np.concatenate(vals[c]), bins=64, range=(0, 255), color="#534AB7")
    ax.flat[k].set_title(c); ax.flat[k].set_yticks([])
plt.tight_layout(); plt.show()

- **Magnetogram**: the histogram is a spike at about 128. Grey 128 = zero field. Normalise with `(x - 128) / 128` so that zero field stays zero and the sign is kept.
- **AIA**: use `(x / 255 - mean) / std` with the per-channel mean and std from the table, computed on training data only.
- Keep the table: these are the numbers that go into `config.yaml`.

## 9. A baseline with no deep learning

Before any CNN: how far do a handful of numbers get? For each sample we take, at the last time step,

- magnetogram: mean distance from grey 128 (how much field), share of very strong pixels
- AIA 94, 131, 171: mean brightness and 99th percentile

then fit a logistic regression, **cross-validated by active region**.

Metrics: **AUC** (ranking quality, 0.5 = random) and **TSS** = recall - false alarm rate (0 = useless, 1 = perfect).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

def features(row):
    f = {}
    a = last_step(row, "magnetogram")
    if a is not None and a.std() > 3:
        d = np.abs(a.astype(float) - 128); f["mag_abs"] = d.mean(); f["mag_strong"] = (d > 100).mean()
    for c in ["94", "131", "171"]:
        a = last_step(row, c)
        if a is not None and a.std() > 3:
            f[f"{c}_mean"] = a.mean(); f[f"{c}_p99"] = np.percentile(a, 99)
    return f

feat = pd.DataFrame([features(r) for r in meta.itertuples()], index=meta.index)
data = pd.concat([meta[["split", "ar", "y", "peak_flux"]], feat], axis=1).dropna()
print(len(data), "samples with all features,", int(data.y.sum()), "positives")

In [ ]:
def tss_at(y, p, thr):
    pred = p >= thr
    tp, fn = (pred & (y == 1)).sum(), (~pred & (y == 1)).sum()
    fp, tn = (pred & (y == 0)).sum(), (~pred & (y == 0)).sum()
    return tp / max(tp + fn, 1) - fp / max(fp + tn, 1)

def best_threshold(y, p):
    grid = np.quantile(p, np.linspace(.02, .98, 97))
    return grid[np.argmax([tss_at(y, p, t) for t in grid])]

SETS = {"magnetogram only": ["mag_abs", "mag_strong"],
        "AIA only": [f"{c}_{s}" for c in ["94", "131", "171"] for s in ["mean", "p99"]]}
SETS["both"] = SETS["magnetogram only"] + SETS["AIA only"]
make = lambda: make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))

train = data[data.split == "training"]; test = data[data.split == "test"]
out = []
for name, cols in SETS.items():
    # out-of-fold predictions on training, folds never share a region
    oof = np.zeros(len(train))
    for tr, va in GroupKFold(5).split(train, train.y, train.ar):
        oof[va] = make().fit(train.iloc[tr][cols], train.iloc[tr].y).predict_proba(train.iloc[va][cols])[:, 1]
    thr = best_threshold(train.y.values, oof)                      # threshold chosen on validation only
    res = {"features": name, "CV AUC": roc_auc_score(train.y, oof), "CV TSS": tss_at(train.y.values, oof, thr)}
    if test.y.nunique() == 2:
        p = make().fit(train[cols], train.y).predict_proba(test[cols])[:, 1]
        res["test AUC"] = roc_auc_score(test.y, p); res["test TSS"] = tss_at(test.y.values, p, thr)
    out.append(res)
pd.DataFrame(out).set_index("features").round(3)

This table is the bar your CNNs have to clear. Note which modality wins before any deep learning: it is a first answer to "HMI vs AIA".

On a real project the test columns are opened **once**, at the end. They are shown here because this baseline is your reference point, not a model you will tune.

## 10. The split you will reuse for every model

Validation folds are built from the `training` folder, grouped by region. The `test` folder stays closed.

In [ ]:
train_meta = meta[meta.split == "training"].reset_index(drop=True)
train_meta["fold"] = -1
from sklearn.model_selection import StratifiedGroupKFold
for k, (_, va) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=0).split(train_meta, train_meta.y, train_meta.ar)):
    train_meta.loc[va, "fold"] = k
summary = train_meta.groupby("fold").agg(samples=("id", "size"), regions=("ar", "nunique"), positives=("y", "sum"))
summary["flaring regions"] = train_meta[train_meta.y == 1].groupby("fold").ar.nunique()
display(summary.fillna(0).astype(int))
assert all(train_meta.groupby("ar").fold.nunique() == 1), "a region is in two folds!"
train_meta[["id", "ar", "peak_flux", "y", "fold"]].to_csv("folds.csv", index=False)
print("saved folds.csv: use the same file for every model")

`StratifiedGroupKFold` keeps each region in one fold and spreads positives as evenly as it can. Check the `positives` column anyway: a few regions own many positives, so folds are never perfectly equal.

## What to write down before training

1. Positives in train / test, and how many regions they come from.
2. Share of samples usable with your chosen channels at the last step.
3. Normalisation constants per channel.
4. Baseline AUC and TSS, for magnetogram only, AIA only, both.